# svg提取数据 重构

## 1. 项目概述

本项目是一个从网页SVG图表中提取数据的工具集，主要功能包括：

1. **SVG图表解析**：从Highcharts生成的SVG图表中提取数据点
2. **多图表类型支持**：支持柱形图、线图等多种图表类型
3. **坐标转换**：将SVG坐标转换为实际数据值
4. **线性插值**：基于已知点进行数值插值
5. **X轴标签生成**：根据起止日期生成时间序列

## 2. 环境配置

In [ ]:
# 导入必要的库
import xml.etree.ElementTree as ET
import re
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import sqlalchemy
import requests
from bs4 import BeautifulSoup
from collections import Counter
import math
import os

# 导入配置
from config import get_database_engine, get_known_points, get_x_axis_config

print("环境配置完成")

## 3. 数据获取

### 3.1 SVG文件读取

SVG文件可以通过以下方式获取：
1. 从网页中直接提取SVG源码
2. 右键另存为SVG文件
3. 使用爬虫从网页中提取

In [ ]:
def read_svg_file(svg_file_path):
    """
    读取SVG文件内容
    
    参数:
        svg_file_path: SVG文件路径
    
    返回:
        SVG文件内容字符串
    """
    with open(svg_file_path, 'r', encoding='utf-8') as f:
        return f.read()


def read_svg_from_string(svg_content):
    """
    从字符串解析SVG
    
    参数:
        svg_content: SVG内容字符串
    
    返回:
        XML根元素
    """
    root = ET.fromstring(svg_content)
    return root


# 示例：读取SVG文件
# svg_content = read_svg_file('chart.svg')
# root = read_svg_from_string(svg_content)

## 4. 数据处理

### 4.1 SVG结构分析

In [ ]:
def analyze_svg_structure(svg_file):
    """
    分析SVG文件结构，返回基本信息
    
    参数:
        svg_file: SVG文件路径或XML根元素
    
    返回:
        包含结构信息的字典
    """
    if isinstance(svg_file, str):
        tree = ET.parse(svg_file)
        root = tree.getroot()
    else:
        root = svg_file
    
    namespaces = {'svg': 'http://www.w3.org/2000/svg'}
    
    # 统计各类元素
    rects = root.findall('.//svg:rect', namespaces)
    paths = root.findall('.//svg:path', namespaces)
    
    return {
        'rect_count': len(rects),
        'path_count': len(paths),
        'chart_type': 'bar' if len(rects) > 100 else 'line'
    }


# 示例
# info = analyze_svg_structure('chart.svg')
# print(f"图表类型: {info['chart_type']}")

## 5. 核心逻辑

### 5.1 SVG解析

In [ ]:
def parse_svg(svg_file):
    """
    解析SVG文件，提取所有path元素的数据
    
    参数:
        svg_file: SVG文件路径
    
    返回:
        list: 包含(path_data, style)元组的列表
    """
    tree = ET.parse(svg_file)
    root = tree.getroot()

    # SVG命名空间
    namespace = {'svg': 'http://www.w3.org/2000/svg'}

    # 查找所有path元素
    paths = root.findall('.//svg:path', namespaces=namespace)

    path_data_list = []
    for path in paths:
        path_data = path.get('d')
        style = path.get('style')
        path_data_list.append((path_data, style))

    return path_data_list

### 5.2 柱形图坐标提取

In [ ]:
def detect_y_zero(svg_file):
    """
    自动检测Y轴零点
    
    参数:
        svg_file: SVG文件路径
    
    返回:
        float: Y轴零点坐标
    """
    tree = ET.parse(svg_file)
    root = tree.getroot()

    namespaces = {'svg': 'http://www.w3.org/2000/svg'}
    rects = root.findall('.//svg:rect', namespaces)

    # 过滤出数据点柱形
    filtered_rects = [
        rect for rect in rects
        if rect.get('class', None) in ['highcharts-point', 'highcharts-point highcharts-negative']
    ]

    # 统计y值出现频率
    y_values = [float(rect.get('y', '0')) for rect in filtered_rects]
    y_counter = Counter(y_values)

    # 返回出现最多的y值（可能是零点）
    if y_counter:
        return y_counter.most_common(1)[0][0]
    return 205  # 默认值


def extract_bar_coordinates(svg_file, y_zero=None, limit=None):
    """
    从柱形图SVG中提取坐标点
    
    参数:
        svg_file: SVG文件路径
        y_zero: Y轴零点坐标，如果为None则自动检测
        limit: 限制返回的点数
    
    返回:
        list: [(x, y_coord), ...] 坐标点列表
    """
    tree = ET.parse(svg_file)
    root = tree.getroot()

    namespaces = {'svg': 'http://www.w3.org/2000/svg'}
    rects = root.findall('.//svg:rect', namespaces)

    # 自动检测Y轴零点
    if y_zero is None:
        y_zero = detect_y_zero(svg_file)

    coordinates = []
    seen_x = set()  # 用于去重

    # 过滤出数据点柱形
    filtered_rects = [
        rect for rect in rects
        if rect.get('class', None) in ['highcharts-point', 'highcharts-point highcharts-negative']
    ]

    for rect in filtered_rects:
        x = float(rect.get('x', '0'))
        y = float(rect.get('y', '0'))
        height = float(rect.get('height', '0'))

        # 去重：每个x只取第一个柱形
        if x in seen_x:
            continue
        seen_x.add(x)

        # 判断正负值
        if y == y_zero:
            # 负值柱形，从零点向下延伸
            y_coord = y_zero + height
        else:
            # 正值柱形，高度即为数据值
            y_coord = height

        coordinates.append((x, y_coord))

    # 按x排序
    coordinates.sort(key=lambda p: p[0])

    # 限制返回数量
    if limit:
        coordinates = coordinates[-limit:]

    return coordinates

### 5.3 线图坐标提取

In [ ]:
def extract_line_coordinates(path_data):
    """
    从SVG path的d属性中提取坐标点
    
    支持的SVG路径命令:
    - M x y: 移动到(x, y)
    - L x y: 直线到(x, y)
    - H x: 水平线到x
    - V y: 垂直线到y
    - C x1 y1 x2 y2 x y: 贝塞尔曲线
    - S x2 y2 x y: 平滑贝塞尔曲线
    - Q x1 y1 x y: 二次贝塞尔曲线
    - T x y: 平滑二次贝塞尔曲线
    - Z: 闭合路径
    
    参数:
        path_data: SVG path的d属性字符串
    
    返回:
        list: [(x, y), ...] 坐标点列表
    """
    # 正则匹配命令和参数
    command_re = re.compile(r'([MLHVCSQTZmlhvcsqtz])([^MLHVCSQTZmlhvcsqtz]*)')
    commands = command_re.findall(path_data)

    coordinates = []
    current_pos = (0, 0)

    for command, params in commands:
        # 分割参数并转换为浮点数
        params = [float(p) for p in re.split(r'[ ,]', params.strip()) if p]

        cmd = command.upper()

        if cmd == 'M' or cmd == 'L':
            # M和L命令：成对坐标
            for i in range(0, len(params), 2):
                x = params[i]
                y = params[i + 1]
                coordinates.append((x, y))
                current_pos = (x, y)

        elif cmd == 'H':
            # H命令：只有x坐标
            x = params[0]
            y = current_pos[1]
            coordinates.append((x, y))
            current_pos = (x, y)

        elif cmd == 'V':
            # V命令：只有y坐标
            x = current_pos[0]
            y = params[0]
            coordinates.append((x, y))
            current_pos = (x, y)

        elif cmd == 'C':
            # C命令：6个参数（控制点+终点）
            for i in range(0, len(params), 6):
                x = params[i + 4]  # 终点x
                y = params[i + 5]  # 终点y
                coordinates.append((x, y))
                current_pos = (x, y)

        elif cmd == 'S':
            # S命令：4个参数
            for i in range(0, len(params), 4):
                x = params[i + 2]
                y = params[i + 3]
                coordinates.append((x, y))
                current_pos = (x, y)

        elif cmd == 'Q':
            # Q命令：4个参数
            for i in range(0, len(params), 4):
                x = params[i + 2]
                y = params[i + 3]
                coordinates.append((x, y))
                current_pos = (x, y)

        elif cmd == 'T':
            # T命令：2个参数
            for i in range(0, len(params), 2):
                x = params[i]
                y = params[i + 1]
                coordinates.append((x, y))
                current_pos = (x, y)

        # Z命令：闭合路径，不需要添加坐标

    return coordinates

### 5.4 线性插值

In [ ]:
def interpolate_value(y, known_points):
    """
    基于已知点进行线性插值
    
    参数:
        y: SVG中的Y坐标
        known_points: [(svg_y1, actual_y1), (svg_y2, actual_y2), (svg_y3, actual_y3)]
                     3个已知点，从小到大排序
    
    返回:
        float: 实际数值
    """
    (y1, actual_y1), (y2, actual_y2), (y3, actual_y3) = known_points

    if y1 <= y <= y2:
        # 在第一个区间，线性插值
        return -((y2 - y) / (y2 - y1)) * (actual_y2 - actual_y1) + actual_y2
    elif y2 < y <= y3:
        # 在第二个区间，线性插值
        return ((y - y2) / (y3 - y2)) * (actual_y3 - actual_y2) + actual_y2

    return None


def interpolate_log_value(y, known_points):
    """
    对数坐标插值
    
    当坐标轴是对数刻度时使用此函数
    
    参数:
        y: SVG中的Y坐标
        known_points: [(svg_y1, actual_y1), (svg_y2, actual_y2), (svg_y3, actual_y3)]
    
    返回:
        float: 实际数值
    """
    (y1, actual_y1), (y2, actual_y2), (y3, actual_y3) = known_points

    # 对实际值取对数
    log_y1 = math.log(actual_y1) if actual_y1 > 0 else 0
    log_y2 = math.log(actual_y2) if actual_y2 > 0 else 0
    log_y3 = math.log(actual_y3) if actual_y3 > 0 else 0

    # 线性插值（在对数空间）
    if y1 <= y <= y2:
        log_actual = -((y2 - y) / (y2 - y1)) * (log_y2 - log_y1) + log_y2
    elif y2 < y <= y3:
        log_actual = ((y - y2) / (y3 - y2)) * (log_y3 - log_y2) + log_y2
    else:
        return None

    # 反变换
    return math.exp(log_actual)

### 5.5 X轴标签生成

In [ ]:
def generate_x_axis_labels(start_year, start_month, end_year, end_month):
    """
    生成月度X轴标签
    
    参数:
        start_year: 起始年份
        start_month: 起始月份
        end_year: 结束年份
        end_month: 结束月份
    
    返回:
        list: 日期标签列表，格式为'YYYY-MM-DD'
    """
    labels = []
    year = start_year
    month = start_month

    while year < end_year or (year == end_year and month <= end_month):
        # 获取当前月的最后一天
        if month == 12:
            next_month_first_day = datetime(year + 1, 1, 1)
        else:
            next_month_first_day = datetime(year, month + 1, 1)

        last_day_of_month = next_month_first_day - timedelta(days=1)
        date_str = last_day_of_month.strftime("%Y-%m-%d")

        labels.append(date_str)

        month += 1
        if month > 12:
            month = 1
            year += 1

    return labels


def generate_quarter_labels(start_year, start_quarter, end_year, end_quarter):
    """
    生成季度X轴标签
    
    参数:
        start_year: 起始年份
        start_quarter: 起始季度(1-4)
        end_year: 结束年份
        end_quarter: 结束季度(1-4)
    
    返回:
        list: 日期标签列表，格式为'YYYY-MM-DD'
    """
    labels = []
    year = start_year
    quarter = start_quarter

    while year < end_year or (year == end_year and quarter <= end_quarter):
        if quarter == 1:
            date_str = f"{year}-03-31"
        elif quarter == 2:
            date_str = f"{year}-06-30"
        elif quarter == 3:
            date_str = f"{year}-09-30"
        elif quarter == 4:
            date_str = f"{year}-12-31"

        labels.append(date_str)

        quarter += 1
        if quarter > 4:
            quarter = 1
            year += 1

    return labels

### 5.6 数据转换

In [ ]:
def convert_to_actual_values(coordinates, x_axis_labels, known_points):
    """
    将SVG坐标转换为实际数据值
    
    参数:
        coordinates: [(x, y), ...] SVG坐标点列表
        x_axis_labels: X轴标签列表
        known_points: [(svg_y, actual_y), ...] 已知点列表
    
    返回:
        list: [(date, value), ...] 实际数据列表
    """
    actual_values = []

    for index, (x, y) in enumerate(coordinates):
        # X轴映射：根据索引直接取标签
        if index < len(x_axis_labels):
            label = x_axis_labels[index]
        else:
            label = f"unknown_{index}"

        # Y轴映射：线性插值
        actual_y = interpolate_value(y, known_points)

        actual_values.append((label, round(actual_y, 2) if actual_y is not None else None))

    return actual_values


def validate_data(actual_values, known_points):
    """
    验证提取的数据是否合理
    
    参数:
        actual_values: [(date, value), ...] 实际数据列表
        known_points: [(svg_y, actual_y), ...] 已知点列表
    
    返回:
        bool: 数据是否合理
    """
    # 检查已知点的插值结果
    for svg_y, actual_y in known_points:
        interpolated = interpolate_value(svg_y, known_points)
        if interpolated is not None:
            error = abs(interpolated - actual_y)
            if error > 0.1:
                print(f"警告: 插值误差较大: {error}")
                return False

    # 检查数据范围
    values = [v for _, v in actual_values if v is not None]
    if not values:
        return False
    
    min_val, max_val = min(values), max(values)

    # 如果范围过大或过小，可能有问题
    if max_val - min_val > 1000:
        print(f"警告: 数据范围可疑过大: {min_val} - {max_val}")
    if max_val - min_val < 0.01:
        print(f"警告: 数据范围可疑过小: {min_val} - {max_val}")

    return True

## 6. 执行与测试

In [ ]:
def extract_svg_data(svg_file, known_points, start_year, start_month, end_year, end_month, chart_type='bar'):
    """
    完整的SVG数据提取流程
    
    参数:
        svg_file: SVG文件路径
        known_points: [(svg_y, actual_y), ...] 已知点列表
        start_year: 起始年份
        start_month: 起始月份
        end_year: 结束年份
        end_month: 结束月份
        chart_type: 图表类型 ('bar' 或 'line')
    
    返回:
        DataFrame: 提取的数据
    """
    # 1. 提取坐标
    if chart_type == 'bar':
        coordinates = extract_bar_coordinates(svg_file)
    else:
        path_data_list = parse_svg(svg_file)
        # 需要选择正确的path索引
        path_data = path_data_list[0][0]  # 默认取第一个
        coordinates = extract_line_coordinates(path_data)

    print(f"提取到 {len(coordinates)} 个坐标点")

    # 2. 生成X轴标签
    x_axis_labels = generate_x_axis_labels(start_year, start_month, end_year, end_month)
    print(f"生成 {len(x_axis_labels)} 个X轴标签")

    # 3. 转换为实际值
    actual_values = convert_to_actual_values(coordinates, x_axis_labels, known_points)

    # 4. 验证数据
    validate_data(actual_values, known_points)

    # 5. 创建DataFrame
    df = pd.DataFrame(actual_values, columns=['dt', 'value'])

    return df


# 示例执行
if __name__ == '__main__':
    # 已知点示例 (svg_y, actual_y)
    known_points = [
        (-68, -18.6),   # 最小值
        (83, 22.7),     # 中间值
        (186, 51.1)     # 最大值
    ]

    # 提取数据
    # df = extract_svg_data(
    #     svg_file='chart.svg',
    #     known_points=known_points,
    #     start_year=1968,
    #     start_month=5,
    #     end_year=2024,
    #     end_month=5,
    #     chart_type='bar'
    # )
    # print(df.head())
    print("SVG数据提取模块已加载")

### 6.1 保存数据

In [ ]:
def save_to_database(df, table_name, engine=None):
    """
    保存数据到数据库
    
    参数:
        df: DataFrame
        table_name: 表名
        engine: SQLAlchemy引擎
    """
    if engine is None:
        engine = get_database_engine()

    df.to_sql(table_name, con=engine, if_exists='replace', index=False)
    print(f"数据已保存到表: {table_name}")


def save_to_excel(df, file_path):
    """
    保存数据到Excel
    
    参数:
        df: DataFrame
        file_path: 输出文件路径
    """
    df.to_excel(file_path, index=False)
    print(f"数据已保存到: {file_path}")

## 7. 工具函数（可复用）

In [ ]:
def detect_chart_type(svg_file):
    """
    自动识别图表类型
    
    参数:
        svg_file: SVG文件路径
    
    返回:
        str: 'bar' 或 'line'
    """
    info = analyze_svg_structure(svg_file)
    return info['chart_type']


def batch_extract(svg_files, config):
    """
    批量提取多个SVG图表
    
    参数:
        svg_files: SVG文件路径列表
        config: 配置字典
    
    返回:
        list: 提取结果列表
    """
    results = []

    for svg_file in svg_files:
        chart_type = detect_chart_type(svg_file)

        df = extract_svg_data(
            svg_file=svg_file,
            known_points=config['known_points'],
            start_year=config['start_year'],
            start_month=config['start_month'],
            end_year=config['end_year'],
            end_month=config['end_month'],
            chart_type=chart_type
        )

        results.append({
            'file': svg_file,
            'data': df
        })

    return results


# 微信理财数据爬取函数
def extract_keywords_and_number(text):
    """
    从文本中提取理财规模数据
    
    参数:
        text: 网页文本
    
    返回:
        tuple: (number, date) 理财规模和日期
    """
    # 查找包含所有关键词且从"截止"开头的子串
    pattern = re.compile(r'(截止.*?理财.*?规模.*?万亿)')
    match = pattern.search(text)

    if match:
        matched_text = match.group(1)
        start_pos = match.start()
        end_pos = start_pos + 50
        extracted_text = text[start_pos:end_pos]

        # 提取第一个"规模"和"万亿"之间的数字
        number_pattern = re.compile(r'规模(.*?)万亿')
        number_match = number_pattern.search(extracted_text)

        number = None
        if number_match:
            potential_numbers = re.findall(r'[\d,\.]+', number_match.group(1))
            if potential_numbers:
                number = potential_numbers[0]

        # 提取日期并格式化
        date_pattern = re.compile(r'(\d{4}年\d{1,2}月\d{1,2}日)')
        date_match = date_pattern.search(extracted_text)

        formatted_date = None
        if date_match:
            date_str = date_match.group(1)
            try:
                date_obj = datetime.strptime(date_str, "%Y年%m月%d日")
                formatted_date = date_obj.strftime("%Y-%m-%d")
            except ValueError:
                pass

        return number, formatted_date

    return None, None


print("所有工具函数已加载")